The RAG process is fixed: every request is used as the basis for a database search for matching documents. The information found in the matching documents is then used to provide a detailed output.
With agents, the LLM agent is given the *option* to perform a database search for answering a given prompt.

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from ingest import load_faq_data, build_index
import json


In [2]:
load_dotenv()
openai_client = OpenAI()

In [3]:
documents = load_faq_data()
index = build_index(documents)

In [4]:
def search(query):
	boost_dict = {"question": 3.0, "section": 0.5}
	filter_dict = {"course": "llm-zoomcamp"}

	return index.search(
		query,
		num_results=5,
		boost_dict=boost_dict,
		filter_dict=filter_dict
  )

In [5]:
def build_context(search_results):
	lines = []

	for doc in search_results:
		lines.append(doc['section'])
		lines.append('Q: ' + doc['question'])
		lines.append('A: ' + doc['answer'])
		lines.append('')

	return '\n'.join(lines).strip()

In [6]:
PROMPT_TEMPLATE = """
	QUESTION: {question}

	CONTEXT:
	{context}
""".strip()

In [7]:
def build_prompt(query, search_results):
	context = build_context(search_results)
	return PROMPT_TEMPLATE.format(
		question=query, context=context
	)

In [8]:
search_tool = {
	"type": "function",
	'function': {
		"name": "search",
		"description": "Search the FAQ database for entries matching the given query.",
		"parameters": {
			"type": "object",
			"properties": {
				"query": {
					"type": "string",
					"description": "Search query text to look up in the course FAQ."
				}
			},
			"required": ["query"],
			"additionalProperties": False
		}
	}
}

In [9]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

In [10]:
response = openai_client.chat.completions.create(
	model="gpt-4o-mini",
	messages=messages,
	tools=[search_tool],
	user="llm-zoomcamp",
	stream=False
)

In [11]:
if response.choices[0].finish_reason == "tool_calls":
	tool_call = response.choices[0].message.tool_calls[0]

	if tool_call.function.name == "search":
		args = json.loads(tool_call.function.arguments)
		search_results = search(**args)

		messages.append(response.choices[0].message.model_dump(exclude_none=True))
		messages.append({
			"role": "tool",
			"tool_call_id": tool_call.id,
			"content": build_context(search_results),
		})

		response = openai_client.chat.completions.create(
			model="gpt-4o-mini",
			messages=messages,
			tools=[search_tool],
			user="llm-zoomcamp",
			stream=False,
		)


In [13]:
response.choices[0].message.content

'Yes, you can still join the course! However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start learning and submitting homework without registering, as registration is only for gauging interest.'

In [14]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 {'role': 'assistant',
  'annotations': [],
  'tool_calls': [{'id': 'call_JhtRHDJZa817mvmQ85C6trD6',
    'function': {'arguments': '{"query":"join course"}', 'name': 'search'},
    'type': 'function'}]},
 {'role': 'tool',
  'tool_call_id': 'call_JhtRHDJZa817mvmQ85C6trD6',
  'content': 'General Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: When will the course be offered next?\nA: Summer 2027.\n\nGeneral Course-Related Questions\nQ: How should I start the course and follow the weekly workflow?\nA: Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](h